# 05 — Generate paper assets

**Purpose.** In this notebook, I use the verified outputs from the earlier notebooks to prepare the final figures and tables for the paper. I do not train, tune, or evaluate any model again here.

**Inputs.** Eight canonical CSV files from `outputs/tables/`: data summary, split summary, target distribution, annual failure rate, final-test metrics, final-test predictions, Logistic Regression coefficients, and Random Forest feature importances.

**Outputs.** Two summary tables in `outputs/tables/` and six paper figures in `outputs/figures/`.

**Run first.** Run Notebooks 01-04 first.

## Imports and project paths

I keep this notebook focused on formatting and visualization. The code below finds the project root, defines the output folders, and sets the model order used consistently across tables and figures.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import MaxNLocator, PercentFormatter
from sklearn.metrics import precision_recall_curve

pd.set_option("display.max_colwidth", 120)


current_directory = Path.cwd().resolve()
PROJECT_ROOT = None

for candidate in [current_directory, *current_directory.parents]:
    if (
        (candidate / "notebooks").is_dir()
        and (candidate / "data").is_dir()
        and (candidate / "README.md").is_file()
    ):
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root. "
        "Run the notebook from the repository root or notebooks directory."
    )

TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
MODEL_ORDER = [
    "Majority-class baseline",
    "Logistic Regression",
    "Decision Tree",
    "Random Forest",
]

LEARNED_MODEL_ORDER = [
    "Logistic Regression",
    "Decision Tree",
    "Random Forest",
]

MODEL_COLORS = {
    "Majority-class baseline": "#8c8c8c",
    "Logistic Regression": "#4c78a8",
    "Decision Tree": "#e15759",
    "Random Forest": "#59a14f",
}

CLASS_COLORS = {"Active": "#4c78a8", "Failed": "#e15759"}
COEFFICIENT_COLORS = {"positive": "#c44e52", "negative": "#4c78a8"}

CANONICAL_INPUTS = {
    "data_summary": TABLES_DIR / "data_summary.csv",
    "split_summary": TABLES_DIR / "split_summary.csv",
    "target_distribution": TABLES_DIR / "target_distribution.csv",
    "annual_failure_rate": TABLES_DIR / "annual_failure_rate.csv",
    "final_test_metrics": TABLES_DIR / "final_test_metrics.csv",
    "final_test_predictions": TABLES_DIR / "final_test_predictions.csv",
    "logistic_coefficients": TABLES_DIR / "logistic_coefficients.csv",
    "random_forest_feature_importance": TABLES_DIR / "random_forest_feature_importance.csv",
}

RETAINED_PAPER_TABLES = {
    "dataset_summary.csv",
    "model_performance_summary.csv",
}

RETAINED_PAPER_FIGURES = {
    "class_balance.png",
    "annual_failure_rate.png",
    "model_performance_comparison.png",
    "precision_recall_models.png",
    "top_logistic_coefficients.png",
    "random_forest_feature_importance.png",
}

FIGURE_DPI = 300

## Load and verify canonical inputs

I use exactly the saved outputs from Notebooks 01-04. These inputs are treated as fixed records; this notebook only formats them for the paper.

In [3]:
missing_inputs = [
    name
    for name, path in CANONICAL_INPUTS.items()
    if not path.exists()
]

if missing_inputs:
    raise FileNotFoundError(f"Missing canonical input files: {missing_inputs}")

canonical_inputs = {
    name: pd.read_csv(path)
    for name, path in CANONICAL_INPUTS.items()
}

data_summary = canonical_inputs["data_summary"]
split_summary = canonical_inputs["split_summary"]
target_distribution = canonical_inputs["target_distribution"]
annual_failure_rate = canonical_inputs["annual_failure_rate"]
final_test_metrics = canonical_inputs["final_test_metrics"]
final_test_predictions = canonical_inputs["final_test_predictions"]
logistic_coefficients = canonical_inputs["logistic_coefficients"]
random_forest_feature_importance = canonical_inputs["random_forest_feature_importance"]

if list(final_test_metrics["model"]) != MODEL_ORDER:
    raise ValueError("Final-test metrics must use the retained model order.")

if set(final_test_predictions["model"].unique()) != set(MODEL_ORDER):
    raise ValueError("Final-test predictions must contain exactly four models.")


if set(target_distribution["status_label"]) != {"alive", "failed"}:
    raise ValueError("Target distribution must contain alive and failed observations.")

if not annual_failure_rate["year"].is_monotonic_increasing:
    raise ValueError("Annual failure-rate data must be sorted by year.")

input_check = pd.DataFrame(
    {
        "input": list(CANONICAL_INPUTS.keys()),
        "path": [str(path.relative_to(PROJECT_ROOT)) for path in CANONICAL_INPUTS.values()],
        "rows": [len(canonical_inputs[name]) for name in CANONICAL_INPUTS],
        "exists": [path.exists() for path in CANONICAL_INPUTS.values()],
    }
)
display(input_check)


,input,path,rows,exists
0,data_summary,outputs/tables/data_summary.csv,1,True
1,split_summary,outputs/tables/split_summary.csv,3,True
2,target_distribution,outputs/tables/target_distribution.csv,2,True
3,annual_failure_rate,outputs/tables/annual_failure_rate.csv,20,True
4,final_test_metrics,outputs/tables/final_test_metrics.csv,4,True
5,final_test_predictions,outputs/tables/final_test_predictions.csv,62776,True
6,logistic_coefficients,outputs/tables/logistic_coefficients.csv,18,True
7,random_forest_feature_importance,outputs/tables/random_forest_feature_importance.csv,18,True


## Create the dataset-summary table

I combine the full-dataset summary with the company-level split summary. The table is intentionally compact so the same values can be moved into the paper without extra calculation.

In [4]:
full_data = data_summary.iloc[0]
split_by_name = split_summary.set_index("split")
train_split = split_by_name.loc["train"]
test_split = split_by_name.loc["test"]

dataset_summary = pd.DataFrame(
    {
        "measure": [
            "Total company-year observations",
            "Unique companies",
            "Minimum year",
            "Maximum year",
            "Financial variables",
            "Failed observations",
            "Failed-observation share",
            "Outer training observations",
            "Outer training companies",
            "Final test observations",
            "Final test companies",
        ],
        "value": [
            f"{int(full_data['n_rows']):,}",
            f"{int(full_data['n_companies']):,}",
            str(int(full_data["min_year"])),
            str(int(full_data["max_year"])),
            str(int(full_data["n_feature_columns"])),
            f"{int(full_data['n_failed_rows']):,}",
            f"{full_data['failure_rate']:.1%}",
            f"{int(train_split['n_rows']):,}",
            f"{int(train_split['n_companies']):,}",
            f"{int(test_split['n_rows']):,}",
            f"{int(test_split['n_companies']):,}",
        ],
    }
)

dataset_summary.to_csv(TABLES_DIR / "dataset_summary.csv", index=False)
display(dataset_summary)


,measure,value
0,Total company-year observations,"78,682"
1,Unique companies,"8,971"
2,Minimum year,1999
3,Maximum year,2018
4,Financial variables,18
5,Failed observations,"5,220"
6,Failed-observation share,6.6%
7,Outer training observations,"62,988"
8,Outer training companies,"7,176"
9,Final test observations,"15,694"


## Create the model-performance table

I keep only the four retained models and the five compact metrics used in the simplified final comparison. The values are rounded to three decimals for direct use in the paper.

In [5]:
performance_columns = ["model", "accuracy", "pr_auc", "precision_failed", "recall_failed", "f1_failed"]
model_performance_summary = (
    final_test_metrics.set_index("model")
    .loc[MODEL_ORDER]
    .reset_index()[performance_columns]
)
metric_columns = [column for column in performance_columns if column != "model"]
model_performance_summary[metric_columns] = model_performance_summary[metric_columns].round(3)

model_performance_summary.to_csv(
    TABLES_DIR / "model_performance_summary.csv",
    index=False,
    float_format="%.3f",
)
display(model_performance_summary)


,model,accuracy,pr_auc,precision_failed,recall_failed,f1_failed
0,Majority-class baseline,0.925,0.075,0.000,0.000,0.000
1,Logistic Regression,0.361,0.151,0.095,0.879,0.171
2,Decision Tree,0.607,0.131,0.113,0.616,0.190
3,Random Forest,0.787,0.172,0.168,0.465,0.247


## Create the class-balance figure

I included the class-distribution figure because failed observations make up only a small share of the dataset. This explains why ordinary accuracy alone is not sufficient for evaluating the models.

In [6]:
class_balance_plot_data = target_distribution.copy()
class_balance_plot_data["company_status"] = class_balance_plot_data["status_label"].map({"alive": "Active", "failed": "Failed"})
class_balance_plot_data = class_balance_plot_data.set_index("company_status").loc[["Active", "Failed"]].reset_index()

fig, ax = plt.subplots(figsize=(6.5, 4.4))
bars = ax.bar(
    class_balance_plot_data["company_status"],
    class_balance_plot_data["count"],
    color=[CLASS_COLORS[label] for label in class_balance_plot_data["company_status"]],
    edgecolor="white",
    linewidth=1.0,
)
ax.set_title("Class Distribution in the Full Dataset")
ax.set_xlabel("Company status")
ax.set_ylabel("Number of company-year observations")
ax.grid(axis="y", alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)

y_offset = class_balance_plot_data["count"].max() * 0.025
for bar, count, share in zip(bars, class_balance_plot_data["count"], class_balance_plot_data["share"], strict=False):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + y_offset,
        f"{int(count):,}\n{share:.1%}",
        ha="center",
        va="bottom",
        fontsize=9,
    )
ax.set_ylim(0, class_balance_plot_data["count"].max() * 1.15)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "class_balance.png", dpi=FIGURE_DPI, bbox_inches="tight")
plt.close(fig)

display(class_balance_plot_data[["company_status", "count", "share"]])


,company_status,count,share
0,Active,73462,0.933657
1,Failed,5220,0.066343


## Create the annual failure-rate figure

I examined the observed failure rate by year because the failed-class share is not constant across the sample period. The vertical axis starts at zero so the figure does not visually exaggerate the decline. I use this figure only as a descriptive overview of the dataset. I do not interpret the decline as a causal economic trend because changes in dataset composition or reporting may also contribute to it. The figure shows the observed target distribution over time. It does not show model predictions, it does not create a future prediction horizon, and the project uses a company-level split rather than a chronological split.


In [7]:
annual_failure_rate_plot_data = annual_failure_rate.sort_values("year").copy()
peak_row = annual_failure_rate_plot_data.loc[annual_failure_rate_plot_data["failure_rate"].idxmax()]
final_row = annual_failure_rate_plot_data.loc[annual_failure_rate_plot_data["year"].idxmax()]
y_max = float(annual_failure_rate_plot_data["failure_rate"].max()) * 1.15

fig, ax = plt.subplots(figsize=(7.4, 4.5))
ax.plot(
    annual_failure_rate_plot_data["year"],
    annual_failure_rate_plot_data["failure_rate"],
    color="#4c78a8",
    marker="o",
    linewidth=2.2,
)
ax.set_title("Observed Annual Failure Rate")
ax.set_xlabel("Year")
ax.set_ylabel("Failure rate")
ax.set_ylim(0, y_max)
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.grid(alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)

ax.annotate(
    f"Peak: {peak_row['failure_rate']:.1%}",
    xy=(peak_row["year"], peak_row["failure_rate"]),
    xytext=(peak_row["year"] + 0.8, peak_row["failure_rate"] - 0.018),
    arrowprops={"arrowstyle": "->", "color": "#666666", "lw": 1.0},
    fontsize=9,
    color="#444444",
)
ax.annotate(
    f"Final year: {final_row['failure_rate']:.1%}",
    xy=(final_row["year"], final_row["failure_rate"]),
    xytext=(final_row["year"] - 6.2, final_row["failure_rate"] + 0.015),
    arrowprops={"arrowstyle": "->", "color": "#666666", "lw": 1.0},
    fontsize=9,
    color="#444444",
)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "annual_failure_rate.png", dpi=FIGURE_DPI, bbox_inches="tight")
plt.close(fig)


display(annual_failure_rate_plot_data.head())


,year,n_observations,n_failed,n_alive,failure_rate
0,1999,5308,380,4928,0.071590
1,2000,5226,404,4822,0.077306
2,2001,4897,414,4483,0.084542
3,2002,4651,414,4237,0.089013
4,2003,4417,415,4002,0.093955


## Create the model-performance comparison figure

The results show the main trade-off clearly. The majority-class baseline has high accuracy but zero failed-class precision, recall, and F1. Logistic Regression has the highest failed-class recall. Random Forest has the highest final-test PR-AUC, precision, and F1. Decision Tree provides intermediate threshold-based performance.

The panels use different horizontal scales because the metrics have different meanings and ranges. Comparisons should be made between models within the same metric panel; bar lengths should not be compared directly across panels. The Random Forest specification was selected using validation PR-AUC before final-test evaluation. The learned models used balanced class weights, and I do not treat their scores as calibrated real-world probabilities because probability calibration was not performed.


In [8]:
metric_labels = {
    "accuracy": "Accuracy",
    "pr_auc": "PR-AUC",
    "precision_failed": "Failed-class precision",
    "recall_failed": "Failed-class recall",
    "f1_failed": "Failed-class F1",
}
metric_limits = {
    "accuracy": 1.00,
    "pr_auc": 0.20,
    "precision_failed": 0.20,
    "recall_failed": 1.00,
    "f1_failed": 0.30,
}
model_performance_plot_data = final_test_metrics.set_index("model").loc[MODEL_ORDER].reset_index()

fig, axes = plt.subplots(2, 3, figsize=(15.5, 7.8))
axes = axes.ravel()
for axis, metric in zip(axes[:5], metric_labels, strict=False):
    values = model_performance_plot_data[metric]
    upper_limit = metric_limits[metric]
    label_padding = upper_limit * 0.025
    y_positions = np.arange(len(model_performance_plot_data))
    axis.barh(
        y_positions,
        values,
        color=[MODEL_COLORS[model] for model in model_performance_plot_data["model"]],
        edgecolor="white",
        linewidth=0.8,
    )
    axis.set_yticks(y_positions)
    axis.set_yticklabels(model_performance_plot_data["model"])
    axis.invert_yaxis()
    axis.set_xlim(0, upper_limit)
    axis.set_title(metric_labels[metric])
    axis.grid(axis="x", alpha=0.25)
    axis.spines[["top", "right"]].set_visible(False)
    for y_position, value in zip(y_positions, values, strict=False):
        label_x = value + label_padding
        if label_x > upper_limit * 0.92:
            axis.text(
                value - label_padding,
                y_position,
                f"{value:.3f}",
                va="center",
                ha="right",
                fontsize=8.5,
                color="white",
            )
        else:
            axis.text(label_x, y_position, f"{value:.3f}", va="center", ha="left", fontsize=8.5)
fig.delaxes(axes[5])
fig.suptitle("Final-Test Performance Across Evaluation Metrics", fontsize=14, y=0.96)
fig.text(
    0.5,
    0.03,
    "Panel scales differ to improve readability; exact values are shown. Accuracy is reported for context, and model selection used validation PR-AUC.",
    ha="center",
    fontsize=9,
    color="#444444",
)
fig.subplots_adjust(left=0.24, right=0.98, top=0.88, bottom=0.13, wspace=0.62, hspace=0.38)
fig.savefig(FIGURES_DIR / "model_performance_comparison.png", dpi=FIGURE_DPI, bbox_inches="tight")
plt.close(fig)

display(model_performance_plot_data[["model", "accuracy", "pr_auc", "precision_failed", "recall_failed", "f1_failed"]])


,model,accuracy,pr_auc,precision_failed,recall_failed,f1_failed
0,Majority-class baseline,0.925003,0.074997,0.000000,0.000000,0.000000
1,Logistic Regression,0.360902,0.151165,0.094680,0.878505,0.170937
2,Decision Tree,0.607238,0.130846,0.112630,0.615973,0.190439
3,Random Forest,0.787371,0.171798,0.168101,0.464741,0.246897


## Create the precision-recall figure

I used precision-recall curves because failed observations are rare. The curves show the trade-off between precision and recall across many possible classification cut-offs. This is not threshold optimization: the reported 0.50 classification metrics remain the standard comparison results, while PR-AUC summarizes ranking performance across cut-offs.

Random Forest has the strongest final-test PR-AUC, and model selection was made earlier using validation PR-AUC.

In [9]:
pr_auc_by_model = final_test_metrics.set_index("model")["pr_auc"].to_dict()
baseline_rows = final_test_predictions[final_test_predictions["model"] == MODEL_ORDER[0]]
failed_prevalence = baseline_rows["actual_failed"].mean()

fig, ax = plt.subplots(figsize=(7.2, 5.2))
curve_rows = []
for model_name in LEARNED_MODEL_ORDER:
    model_rows = final_test_predictions[final_test_predictions["model"] == model_name]
    precision, recall, _ = precision_recall_curve(
        model_rows["actual_failed"],
        model_rows["probability_failed"],
    )
    ax.step(
        recall,
        precision,
        where="post",
        color=MODEL_COLORS[model_name],
        linewidth=2.0,
        label=f"{model_name} (PR-AUC {pr_auc_by_model[model_name]:.3f})",
    )
    curve_rows.append({"model": model_name, "curve_points": len(precision), "pr_auc": pr_auc_by_model[model_name]})

ax.axhline(
    failed_prevalence,
    color="#666666",
    linestyle="--",
    linewidth=1.2,
    label=f"Failed prevalence ({failed_prevalence:.1%})",
)
ax.set_title("Final-Test Precision-Recall Curves")
ax.set_xlabel("Recall of failed observations")
ax.set_ylabel("Precision for failed observations")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="upper right")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "precision_recall_models.png", dpi=FIGURE_DPI, bbox_inches="tight")
plt.close(fig)

precision_recall_curve_summary = pd.DataFrame(curve_rows)


display(precision_recall_curve_summary)


,model,curve_points,pr_auc
0,Logistic Regression,15663,0.151165
1,Decision Tree,33,0.130846
2,Random Forest,15596,0.171798


## Create the Logistic Regression coefficient figure

The coefficient signs show predictive direction, but they should not be interpreted as causal effects. Positive coefficients are associated with a higher predicted failure score, and negative coefficients are associated with a lower predicted failure score. The financial variables were standardized before Logistic Regression, so coefficient magnitudes are more comparable across variables.

In [10]:
top_logistic_coefficients = (
    logistic_coefficients.nlargest(8, "absolute_coefficient")
    .sort_values("coefficient")
    .reset_index(drop=True)
)
colors = [
    COEFFICIENT_COLORS["positive"] if value > 0 else COEFFICIENT_COLORS["negative"]
    for value in top_logistic_coefficients["coefficient"]
]

fig, ax = plt.subplots(figsize=(7.6, 5.0))
y_positions = np.arange(len(top_logistic_coefficients))
ax.barh(y_positions, top_logistic_coefficients["coefficient"], color=colors, edgecolor="white", linewidth=0.8)
ax.set_yticks(y_positions)
ax.set_yticklabels(top_logistic_coefficients["readable_name"])
ax.axvline(0, color="#333333", linewidth=1.0)
ax.set_title("Largest Logistic Regression Coefficients")
ax.set_xlabel("Standardized coefficient")
ax.grid(axis="x", alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)

max_abs = top_logistic_coefficients["coefficient"].abs().max()
ax.set_xlim(-max_abs * 1.25, max_abs * 1.25)
for y_position, value in zip(y_positions, top_logistic_coefficients["coefficient"], strict=False):
    offset = max_abs * 0.035
    ax.text(
        value + offset if value >= 0 else value - offset,
        y_position,
        f"{value:.2f}",
        va="center",
        ha="left" if value >= 0 else "right",
        fontsize=8.5,
    )
fig.tight_layout()
fig.savefig(FIGURES_DIR / "top_logistic_coefficients.png", dpi=FIGURE_DPI, bbox_inches="tight")
plt.close(fig)


display(top_logistic_coefficients[["feature", "readable_name", "coefficient", "direction"]])


,feature,readable_name,coefficient,direction
0,X8,Market value,-2.461954,lower predicted failure probability
1,X1,Current assets,-2.028779,lower predicted failure probability
2,X12,EBIT,-0.935706,lower predicted failure probability
3,X4,EBITDA,-0.487864,lower predicted failure probability
4,X13,Gross profit,0.433979,higher predicted failure probability
5,X5,Inventory,0.473374,higher predicted failure probability
6,X3,Depreciation and amortization,0.659623,higher predicted failure probability
7,X14,Total current liabilities,0.917720,higher predicted failure probability


## Create the Random Forest importance figure

The Random Forest importance values show how strongly the fitted forest used each variable in its splits. They do not show positive or negative direction, they do not represent causal effects, and importance can be distributed across correlated variables. I interpret this figure as model usage, not as an economic causal conclusion.

In [11]:
random_forest_importance_plot_data = (
    random_forest_feature_importance.nlargest(8, "importance")
    .sort_values("importance")
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(7.4, 5.0))
y_positions = np.arange(len(random_forest_importance_plot_data))
ax.barh(y_positions, random_forest_importance_plot_data["importance"], color=MODEL_COLORS["Random Forest"], edgecolor="white", linewidth=0.8)
ax.set_yticks(y_positions)
ax.set_yticklabels(random_forest_importance_plot_data["readable_name"])
ax.set_title("Top Random Forest Feature Importances")
ax.set_xlabel("Importance")
ax.grid(axis="x", alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)

max_value = random_forest_importance_plot_data["importance"].max()
ax.set_xlim(0, max_value * 1.22)
for y_position, value in zip(y_positions, random_forest_importance_plot_data["importance"], strict=False):
    ax.text(value + max_value * 0.025, y_position, f"{value:.3f}", va="center", fontsize=8.5)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "random_forest_feature_importance.png", dpi=FIGURE_DPI, bbox_inches="tight")
plt.close(fig)

top_random_forest_importance = (
    random_forest_importance_plot_data
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)


display(top_random_forest_importance[["feature", "readable_name", "importance"]])


,feature,readable_name,importance
0,X8,Market value,0.106478
1,X6,Net income,0.085597
2,X3,Depreciation and amortization,0.064994
3,X1,Current assets,0.062450
4,X11,Total long-term debt,0.062069
5,X15,Retained earnings,0.060380
6,X17,Total liabilities,0.056898
7,X7,Total receivables,0.056727


## Verify all generated assets

I finish by checking that the two summary tables exist inside `outputs/tables/`, that the figure folder contains exactly the six retained figures, and that every final asset is non-empty.

In [12]:
table_files = {
    path.name
    for path in TABLES_DIR.iterdir()
    if path.is_file()
}

figure_files = {
    path.name
    for path in FIGURES_DIR.iterdir()
    if path.is_file()
}

missing_summary_tables = RETAINED_PAPER_TABLES - table_files

if missing_summary_tables:
    raise FileNotFoundError(f"Missing summary tables: {sorted(missing_summary_tables)}")

if figure_files != RETAINED_PAPER_FIGURES:
    raise ValueError(
        "The final figure folder does not contain exactly "
        "the six retained figures. "
        f"Missing: {sorted(RETAINED_PAPER_FIGURES - figure_files)}; "
        f"unexpected: {sorted(figure_files - RETAINED_PAPER_FIGURES)}"
    )


asset_rows = []
for folder_name, folder, retained_names in [("outputs/tables", TABLES_DIR, RETAINED_PAPER_TABLES), ("outputs/figures", FIGURES_DIR, RETAINED_PAPER_FIGURES)]:
    for file_name in sorted(retained_names):
        path = folder / file_name
        asset_rows.append(
            {
                "folder": folder_name,
                "file": file_name,
                "size_bytes": path.stat().st_size,
                "non_empty": path.stat().st_size > 0,
            }
        )
asset_check = pd.DataFrame(asset_rows).sort_values(["folder", "file"]).reset_index(drop=True)

if not asset_check["non_empty"].all():
    raise ValueError("Every generated final asset must be non-empty.")

display(asset_check)


,folder,file,size_bytes,non_empty
0,outputs/figures,annual_failure_rate.png,127377,True
1,outputs/figures,class_balance.png,85186,True
2,outputs/figures,model_performance_comparison.png,311113,True
3,outputs/figures,precision_recall_models.png,153206,True
4,outputs/figures,random_forest_feature_importance.png,110380,True
5,outputs/figures,top_logistic_coefficients.png,98263,True
6,outputs/tables,dataset_summary.csv,329,True
7,outputs/tables,model_performance_summary.csv,255,True


## Final summary

- In this notebook, I used the verified outputs from the earlier notebooks to prepare the final paper assets.
- I did not train, tune, refit, or evaluate any model again here.
- I created a compact dataset summary table and a four-model performance summary table.
- I created six figures: class balance, annual observed failure rate, final-test metric comparison, precision-recall curves, Logistic Regression coefficients, and Random Forest feature importances.
- The figures keep the interpretation careful: model scores are not calibrated real-world probabilities, coefficient signs are predictive associations rather than causal effects, and Random Forest importance shows model usage rather than economic causality.